# Notebook08 data preparation and actual-step calibration
Tagged definitions only. Reuse the sealed Food101 training/development membership and the exact Notebook07 pilot configuration; official test remains reserved. Calibration runs8 discarded updates per model with fixed physical/effective batch128. It checks feasibility of the fixed10-pair350-update protocol; it never selects a smaller study or changes the architecture. Expected runtime2–4minutes. Outputs: artifacts/notebook_08/<run_id>/stages/learning/. Checkpoints: checkpoints/<run_id>/calibration/. Resume using unchanged main Notebook08 before its original8-hour deadline.

In [ ]:
"""Food101 membership verification and bounded actual-update calibration for Notebook08."""
from pathlib import Path
import copy, gc, hashlib, importlib.util, json, statistics, sys, time
import torch
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json, digest_file
from fmm_jepa.food101_training import isolated_rng, validate_training_manifest
from fmm_jepa.semantic_pilot import configure_environment

def load_module(root, name, folder="08_replication", prefix="fmm08_"):
    key=prefix+name
    if key not in sys.modules:
        path=Path(root)/"experiments"/folder/(name+".ipynb")
        spec=importlib.util.spec_from_loader(key, TaggedNotebookLoader(path))
        module=importlib.util.module_from_spec(spec);sys.modules[key]=module
        try:spec.loader.exec_module(module)
        except BaseException:
            sys.modules.pop(key,None);raise
    return sys.modules[key]

def canonical(value):
    return json.loads(json.dumps(value,allow_nan=False))

def digest(value):
    return hashlib.sha256(json.dumps(canonical(value),sort_keys=True,separators=(',',':')).encode()).hexdigest()

def immutable(path, value):
    path=Path(path);value=canonical(value)
    if path.exists():
        if json.loads(path.read_text())!=value:raise ValueError('Immutable evidence changed: '+str(path))
    else:atomic_json(path,value)
    return value

def prepare_bundle(root, artifact, config):
    root,artifact=Path(root).resolve(),Path(artifact).resolve()
    module=load_module(root,'preparation','05_scaling','fmm08_data05_')
    def paths(source_root, folder, stage):
        source_root,folder=Path(source_root).resolve(),Path(folder).resolve()
        run_id=folder.parent.parent.name
        if folder!=source_root/'artifacts/notebook_08'/run_id/'stages'/'learning':
            raise ValueError('Unexpected Notebook08 data path')
        return folder,source_root/'checkpoints'/run_id/'stages'/'learning'/'calibration',run_id
    module._stage_paths=paths
    return module.prepare_stage_bundle(root,artifact,{'id':'learning','image_size':128,'pretrain_pool':'selected'},config,verify_images=True)

def base_config(root):
    old=json.loads((Path(root)/'artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json').read_text())
    cfg=copy.deepcopy(old['arms'][0]['config'])
    cfg.update(device='cuda:0',dtype='float32',image_size=128,numerical_backend='energy08-v1')
    cfg['training'].update(updates=350,checkpoint_every=50,microbatch=128,accumulation_steps=1,max_consecutive_failures=1)
    cfg['probe'].update(epochs=20,selection='fixed_final_epoch',standardize=True)
    cfg['evaluation'].update(heldout_enabled=False,batch_size=64,diagnostic_steps=[0,350])
    cfg['budget'].update(max_seconds=28800,evaluation_reserve_seconds=4000)
    cfg['resolution_rationale']='Notebook08 energy encoder versus ViT at128px; common patch8/width192/predictor; distinct encoder parameter counts explicitly reported.'
    cfg['comparison']='Paired seeds, equal updates/exposures; measured compute differs; final probe epoch20.'
    cfg['failure_policy']='Stop on first numerical failure; never skip a consumed batch.'
    cfg['seeds'].update(model_init=20269901,data_order=611101,augmentation=622202,mask=633303,probe=4404)
    cfg['model']['feature_seed']=20270901
    cfg['model'].update(energy_summary_dim=32,energy_steps=4,energy_near_strength=.25,energy_far_strength=.25,energy_step_size=.5,energy_coefficient_dim=32,energy_checkpoint=True)
    return configure_environment(cfg)

def arm_config(config, variant, feature_counts=None):
    cfg=copy.deepcopy(config)
    if variant not in ('vit','energy'):raise ValueError('Unknown energy-study arm')
    cfg['energy07']={'variant':variant}
    return cfg

def _release(device):
    gc.collect()
    if torch.device(device).type=='cuda':torch.cuda.synchronize(device);torch.cuda.empty_cache()

def _trace_hash(traces):
    return digest([{k:t[k] for k in ('attempted_update','sample_ids','occurrences','augmentation_ids','mask_ids','successful','successful_update')} for t in traces])

def calibrate(root, artifact, config, bundle, deadline_unix):
    """Eight actual updates per arm; 3warmups +5timings; all weights discarded."""
    root,artifact=Path(root).resolve(),Path(artifact).resolve()
    runner=load_module(root,'runner')
    out=artifact/'calibration_result.json';folder=artifact/'calibration';folder.mkdir(exist_ok=True)
    run_id=artifact.parent.parent.name
    identity={'config':config,'source_hashes':runner.stage_definition_hashes(root),
              'split_sha256':bundle['split_manifest']['split_sha256'],'microbatch_candidates':[128],
              'variants':['vit','energy'],'warmup':3,'measured':5,'peak_limit_bytes':70*1024**3}
    identity=canonical(identity);identity_hash=digest(identity)
    immutable(folder/'protocol.json',{'identity':identity,'identity_hash':identity_hash})
    if out.exists():
        result=json.loads(out.read_text())
        if result['identity_hash']!=identity_hash:raise ValueError('Calibration identity changed')
        for row in result['rows']:
            if row['status']=='PASS':validate_training_manifest(row['checkpoint_dir'])
        return result
    rows=[];reference_trace=None;selected=None
    for microbatch in identity['microbatch_candidates']:
        current=[]
        for variant in identity['variants']:
            if time.time()>=deadline_unix-30:raise TimeoutError('Calibration deadline')
            key=f'{variant}_mb{microbatch}';case=folder/key;case.mkdir(exist_ok=True)
            checkpoint=root/'checkpoints'/run_id/'calibration'/key
            cfg=arm_config(config,variant)
            cfg['training'].update(microbatch=microbatch,accumulation_steps=128//microbatch,checkpoint_every=4)
            kind='vit' if variant=='vit' else 'semantic'
            row_path=case/'case.json'
            if row_path.exists():
                row=json.loads(row_path.read_text())
                if row['identity_hash']!=identity_hash:raise ValueError('Calibration case changed')
            else:
                model=result=None
                row={'variant':variant,'kind':kind,'microbatch':microbatch,'identity_hash':identity_hash,'checkpoint_dir':str(checkpoint)}
                try:
                    _release(cfg['device'])
                    with isolated_rng(cfg['seeds']['model_init']):model=runner.build_model(cfg,kind).cpu()
                    plan={'run_id':run_id,'stage_id':'learning','updates':350,'milestones':[350],
                          'resume_notebook':'notebooks/08_Energy_Encoder_Replication.ipynb','resume_cell':'launch-energy-replication'}
                    protocol={'source_hashes':identity['source_hashes'],'calibration_identity_hash':identity_hash,
                              'purpose':'Notebook08 actual-update calibration; weights discarded'}
                    result=runner._run_stage_model(plan,kind,cfg,bundle,case,checkpoint,protocol,model=model,stop_after=8,deadline_unix=deadline_unix-15)
                    if result['resume']['final_step']!=8 or len(result['traces'])!=8 or any(not t['successful'] for t in result['traces']):
                        raise RuntimeError('Eight clean calibration updates required')
                    measured=result['metrics'][3:8]
                    peak=int(result['performance']['peak_cuda_allocated_bytes'])
                    row.update(status='PASS' if peak<=identity['peak_limit_bytes'] else 'MEMORY_LIMIT',
                               median_step_seconds=statistics.median(x['step_seconds'] for x in measured),
                               raw_step_seconds=[x['step_seconds'] for x in measured],peak_cuda_allocated_bytes=peak,
                               trace_sha256=_trace_hash(result['traces']),successful_updates=8,failed_attempts=0)
                except torch.cuda.OutOfMemoryError as error:row.update(status='OOM',error=str(error))
                finally:
                    model=result=None;_release(cfg['device'])
                immutable(row_path,row)
            if row['status']=='PASS':
                validate_training_manifest(checkpoint)
                traces=json.loads((case/(kind+'_training_traces.json')).read_text())
                signature=_trace_hash(traces)
                if signature!=row['trace_sha256']:raise ValueError('Calibration traces changed')
                if reference_trace is None:reference_trace=signature
                if reference_trace!=signature:raise AssertionError('Calibration samples/augmentations/masks differ')
            rows.append(row);current.append(row)
            print('Calibration',key,row['status'],row.get('median_step_seconds'),flush=True)
        if all(r['status']=='PASS' for r in current):selected=microbatch;break
    if selected is None:raise RuntimeError('No common feasible batch fits70GiB')
    accumulation_certificate={'status':'NOT_APPLICABLE','reason':'Physical batch equals effective batch; no gradient accumulation.'}
    if selected<128:
        engineering=load_module(root,'engineering','04_followup','fmm08_accum04_')
        engineering.initialize_models=runner.initialize_models
        certificates={}
        for variant in identity['variants']:
            kind='vit' if variant=='vit' else 'semantic'
            engineering.ENGINEERING_KINDS=(kind,)
            cfg=arm_config(config,variant)
            destination=folder/'gradient_certificate'/variant
            destination.mkdir(parents=True,exist_ok=True)
            immutable(destination/'binding.json',{'identity_hash':identity_hash,'config':cfg,'selected_microbatch':selected,'variant':variant})
            certificates[variant]=engineering.certify_microbatch_equivalence(root,destination,cfg,bundle,selected,deadline_unix)
            if certificates[variant]['status']!='PASS':raise AssertionError('Candidate accumulation equivalence failed')
        accumulation_certificate={'status':'PASS','variants':certificates,'baseline_microbatch':16,'selected_microbatch':selected}
    result={'status':'PASS','identity_hash':identity_hash,'rows':rows,'selected_microbatch':selected,
            'selected_accumulation_steps':128//selected,'selected_model_step_seconds':{r['variant']:r['median_step_seconds'] for r in current},
            'selected_model_peak_bytes':{r['variant']:r['peak_cuda_allocated_bytes'] for r in current},
            'common_trace_sha256':reference_trace,'accumulation_certificate':accumulation_certificate,'weights_discarded':True,'quality_evidence':False,
            'timing_scope':'Median of5complete actual-image updates after3warmups; FP32, data included, checkpoint I/O excluded.'}
    immutable(out,result)
    return result
